# 跳包模型

这里是跳包模型的完整实现。我们要把那个“玩具模型”扔进垃圾桶，因为它有两个致命问题，导致无法处理数百万词汇量的真实语料：

1. Softmax 计算量太大：每次预测都要计算几万个词的概率，词表一大，训练速度会慢到让你怀疑人生。
2. 高频词干扰：像 "the" 这种词出现几百万次，如果不处理，模型会花费大量时间去学 "the" 和其他词的关系，既浪费时间又拉低精度。

我们要实现的是 带有负采样 (Negative Sampling) 和 下采样 (Sub-sampling) 的 Skip-gram。这是 Google 在 Word2Vec 论文中使用的真正架构。

我们将使用 WikiText-2 数据集（维基百科文章集合），这是一个标准的 NLP 工业级基准数据集。

## 第一步：环境配置与数据下载

我们需要处理真实文件。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import collections
import random
import math
import time

# ============================================================
# 检查并选择计算设备
# ============================================================
# 优先使用 GPU（CUDA），如果不可用则使用 CPU
# GPU 可以显著加速矩阵运算，对于大规模训练至关重要
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 性能对比：
# - CPU: 适合小规模实验和调试
# - GPU: 大规模训练速度可提升 10-100 倍


Using device: cuda


获取数据：

In [2]:
!wget https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/train.txt

--2026-02-15 09:42:52--  https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/train.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10797148 (10M) [text/plain]
Saving to: ‘train.txt’

train.txt           100%[===================>]  10.30M  --.-KB/s    in 0.02s   

2026-02-15 09:42:53 (468 MB/s) - ‘train.txt’ saved [10797148/10797148]



In [ ]:
# ============================================================
# 读取 WikiText-2 训练数据
# ============================================================
# WikiText-2 是一个标准的语言模型基准数据集
# 包含来自维基百科的高质量文章，总词数约 200 万
# 相比于玩具数据集，这是真实的工业级语料库

with open('train.txt', 'r') as f:
    text_data = f.read()
    
print(f"数据加载完成，字符长度: {len(text_data)}")
# 预期输出：数百万个字符


数据加载完成，字符长度: 10780437


## 第二步：工业级预处理 (Sub-sampling)

### 高频词下采样 (Sub-sampling):

我们要以此公式丢弃高频词：$P(w_i) = 1 - \sqrt{\frac{t}{f(w_i)}}$

其中 $t$ 是阈值（通常 $10^{-5}$），$f(w_i)$ 是词频。

这能让模型少学几次 "the"，多学几次 "learning"。

In [ ]:
class Vocab:
    """工业级词表类，支持低频词过滤和高频词下采样
    
    关键特性：
    1. 低频词过滤：移除出现次数少于 min_freq 的词，减少噪声
    2. 高频词下采样：降低 "the", "is" 等高频词的出现概率
    """
    
    def __init__(self, text, min_freq=5):
        """初始化词表
        
        参数:
            text: 原始文本数据
            min_freq: 最小词频阈值，低于此值的词会被过滤
        """
        # 步骤1: 分词（简单的空格分割）
        tokens = text.lower().split()
        
        # 步骤2: 统计词频
        # Counter 自动统计每个词出现的次数
        self.counter = collections.Counter(tokens)
        
        # 步骤3: 过滤低频词
        # 工业界通常过滤掉出现少于 5 次的词，因为：
        #   - 这些词可能是拼写错误或专有名词
        #   - 样本太少无法学到有意义的向量
        #   - 可以显著减小词表大小，节省内存和计算
        self.tokens = [token for token in tokens if self.counter[token] >= min_freq]
        
        # 步骤4: 构建词表
        self.unique_tokens = sorted(list(set(self.tokens)))
        
        # 步骤5: 创建词 <-> 索引 的双向映射
        self.word_to_idx = {word: i for i, word in enumerate(self.unique_tokens)}
        self.idx_to_word = {i: word for i, word in enumerate(self.unique_tokens)}
        self.vocab_size = len(self.unique_tokens)
        
        # 步骤6: 计算词频（归一化）
        # 用于下采样计算
        total_count = len(self.tokens)
        self.freqs = {w: c/total_count for w, c in self.counter.items()}

    def subsample(self, t=1e-5):
        """高频词下采样
        
        问题：在自然语言中，"the", "is", "a" 等词出现频率极高
        影响：
        - 训练时会反复学习这些词，浪费计算资源
        - 这些词的语义信息较少，对模型质量贡献不大
        
        解决方案：以一定概率丢弃高频词
        
        丢弃概率公式（Word2Vec 论文）：
            P(w_i) = 1 - sqrt(t / f(w_i))
        
        其中：
            t: 阈值参数（通常为 1e-5）
            f(w_i): 词 w_i 的频率
        
        特性：
            - 频率越高，丢弃概率越大
            - 频率很低的词几乎不会被丢弃
            - 如 "the" (频率 0.05) 可能有 95% 概率被丢弃
            - 如 "learning" (频率 0.0001) 几乎不会被丢弃
        
        参数:
            t: 下采样阈值，控制丢弃的激进程度
        """
        # 计算每个词的丢弃概率
        drop_prob = {w: 1 - math.sqrt(t / self.freqs[w]) for w in self.unique_tokens}
        
        # 以概率 drop_prob 丢弃词
        # random.random() 生成 [0,1) 的随机数
        # 如果随机数 > 丢弃概率，则保留这个词
        self.train_tokens = [w for w in self.tokens if random.random() > drop_prob.get(w, 0)]
        
        print(f"下采样前词数: {len(self.tokens)}")
        print(f"下采样后词数: {len(self.train_tokens)}")
        # 预期：词数减少 10-30%，大部分是高频词

# ============================================================
# 初始化词表并执行下采样
# ============================================================
# min_freq=1: 因为这是演示数据，样本较少，设为 1
# 在真实的大规模语料中，通常设为 5-10
vocab = Vocab(text_data, min_freq=1)
vocab.subsample()


下采样前词数: 2051910
下采样后词数: 502247


## 第三步：构建负采样数据集 (Negative Sampling Dataset)

这是最核心的变化。我们不再生成 (center, context),而是生成：(center, context, negatives)。

* Center: fox
* Context (正样本): jumps (标签为 1)
* Negatives (负样本): apple, car, sky... (标签为 0)

In [ ]:
class Word2VecDataset(Dataset):
    """负采样 Skip-gram 数据集
    
    与简单版本的关键区别：
    1. 不是生成 (center, context) 对
    2. 而是生成 (center, positive_context, negative_contexts)
    
    概念：
    - Positive Sample（正样本）：真实的上下文词，标签为 1
    - Negative Samples（负样本）：随机采样的非上下文词，标签为 0
    
    为什么用负采样？
    - 传统 Softmax：需要计算整个词表的概率分布，复杂度 O(V)，V 是词表大小
    - 负采样：只需要计算 1 个正样本 + K 个负样本，复杂度 O(K)，K 通常是 5-20
    - 速度提升：100 倍以上（当词表有 10 万词时）
    """
    
    def __init__(self, tokens, word_to_idx, vocab_size, window_size=3, num_negatives=5):
        """初始化数据集
        
        参数:
            tokens: 词列表（已经过下采样）
            word_to_idx: 词到索引的映射
            vocab_size: 词表大小
            window_size: 上下文窗口大小
            num_negatives: 每个正样本对应的负样本数量
        """
        # 将词转换为索引
        self.tokens = [word_to_idx[w] for w in tokens]
        self.window_size = window_size
        self.num_negatives = num_negatives
        self.vocab_size = vocab_size
        
        # ========================================================
        # 预先计算负采样权重（重要优化）
        # ========================================================
        # 负采样不是均匀随机的，而是按词频加权
        # 权重公式：P(w) ∝ f(w)^0.75
        # 
        # 为什么用 0.75 次方？（Word2Vec 论文的经验值）
        # - 纯词频 f(w)^1.0：高频词被采样太多次
        # - 均匀分布 f(w)^0.0：低频词被过度采样
        # - f(w)^0.75：折中方案，效果最好
        #
        # 示例：
        #   "the" 频率 0.05 -> 权重 0.05^0.75 ≈ 0.084
        #   "cat" 频率 0.001 -> 权重 0.001^0.75 ≈ 0.0056
        #   相比直接用频率，降低了高频词的采样概率
        
        # 步骤1: 获取每个词的出现次数
        word_counts = np.array([vocab.counter[vocab.idx_to_word[i]] for i in range(vocab_size)])
        
        # 步骤2: 计算词频
        word_freqs = word_counts / np.sum(word_counts)
        
        # 步骤3: 应用 0.75 次方，并转为 PyTorch 张量
        self.neg_sample_weights = torch.tensor(word_freqs ** 0.75)

    def __len__(self):
        """返回数据集大小"""
        return len(self.tokens)

    def __getitem__(self, idx):
        """获取一个训练样本
        
        返回:
            center_word: 中心词索引
            pos_word: 正样本（上下文词）索引
            neg_words: 负样本索引列表
        """
        # 步骤1: 获取中心词
        center_word = self.tokens[idx]
        
        # 步骤2: 获取正样本（上下文词）
        # 定义窗口范围
        start = max(0, idx - self.window_size)
        end = min(len(self.tokens), idx + self.window_size + 1)
        
        # 收集窗口内的所有词（排除中心词自己）
        context_words = self.tokens[start:idx] + self.tokens[idx+1:end]
        
        # 边界情况处理：如果没有上下文词（不太可能），跳到下一个
        if len(context_words) == 0:
            return self.__getitem__((idx + 1) % len(self.tokens))
        
        # 工业界优化：随机选择窗口内的一个词作为正样本
        # 而不是遍历所有上下文词，这样可以加速训练
        pos_word = random.choice(context_words)
        
        # 步骤3: 采样负样本
        # torch.multinomial: 按权重进行多项式采样
        # replacement=True: 允许重复采样（同一个词可能被选中多次）
        neg_words = torch.multinomial(
            self.neg_sample_weights, 
            self.num_negatives, 
            replacement=True
        )
        
        return torch.tensor(center_word), torch.tensor(pos_word), neg_words

# ============================================================
# 创建数据加载器
# ============================================================
dataset = Word2VecDataset(vocab.train_tokens, vocab.word_to_idx, vocab.vocab_size)

# DataLoader: PyTorch 的数据加载工具
# - batch_size=512: 每次训练使用 512 个样本（较大的批次可以提高 GPU 利用率）
# - shuffle=True: 每个 epoch 打乱数据顺序，避免过拟合
# - num_workers=4: 使用 4 个进程并行加载数据，加速训练
dataloader = DataLoader(dataset, batch_size=512, shuffle=True, num_workers=4)


## 第四步：实现 Skip-gram 模型

之前的模型用的是 nn.Linear + Softmax。现在的模型用的是 双 Embedding 层 + LogSigmoid。

我们需要两个 Embedding 矩阵：

* in_embed:用来查中心词。
* out_embed: 用来查上下文词（和负样本词）。

这段代码利用了矩阵乘法 (torch.bmm) 并行计算所有样本的相似度。我们不再计算整个词表的 Softmax，只计算 1个正样本 + 5个负样本 的 Sigmoid。这就是让训练速度提升 1000 倍的秘诀。

In [ ]:
class SkipGramNegSampling(nn.Module):
    """带负采样的 Skip-gram 模型（工业级实现）
    
    关键创新：
    1. 双 Embedding 矩阵：in_embed（中心词）和 out_embed（上下文词）
    2. 负采样损失：不计算全词表 Softmax，只优化正负样本
    3. Sigmoid 激活：将相似度转换为二分类概率
    
    为什么需要两个 Embedding？
    - 在经典的 Word2Vec 实现中，每个词同时扮演两个角色：
      * 作为中心词时，使用 in_embed
      * 作为上下文词时，使用 out_embed
    - 理论上可以共享，但实践中分开效果更好且训练更稳定
    """
    
    def __init__(self, vocab_size, embed_dim):
        """初始化模型
        
        参数:
            vocab_size: 词表大小
            embed_dim: 词向量维度（工业界常用 100-300）
        """
        super(SkipGramNegSampling, self).__init__()
        
        # ========================================================
        # 两个 Embedding 矩阵
        # ========================================================
        # in_embed: 中心词向量矩阵，形状 (vocab_size, embed_dim)
        # 这是我们最终要使用的词向量
        self.in_embed = nn.Embedding(vocab_size, embed_dim)
        
        # out_embed: 上下文词向量矩阵（也叫输出矩阵）
        # 形状同样是 (vocab_size, embed_dim)
        # 训练完成后通常只保留 in_embed，丢弃 out_embed
        self.out_embed = nn.Embedding(vocab_size, embed_dim)
        
        # ========================================================
        # 权重初始化（非常重要）
        # ========================================================
        # 使用小范围的均匀分布初始化
        # 范围: [-0.5/embed_dim, 0.5/embed_dim]
        # 好处：
        # 1. 打破对称性：如果所有权重相同，网络无法学习
        # 2. 防止梯度爆炸/消失：初始值太大或太小都会影响训练
        # 3. 加速收敛：合适的初始化可以让训练更快
        self.in_embed.weight.data.uniform_(-0.5 / embed_dim, 0.5 / embed_dim)
        self.out_embed.weight.data.uniform_(-0.5 / embed_dim, 0.5 / embed_dim)

    def forward(self, center, context, negatives):
        """前向传播：计算负采样损失
        
        理论基础（Word2Vec 负采样目标）：
        最大化：log σ(v_c · v_o) + Σ log σ(-v_c · v_n)
        
        其中：
        - v_c: 中心词向量
        - v_o: 正样本（真实上下文词）向量
        - v_n: 负样本向量
        - σ: Sigmoid 函数
        
        直观理解：
        - 让中心词和真实上下文词的向量点积尽可能大（相似）
        - 让中心词和负样本的向量点积尽可能小（不相似）
        
        参数:
            center: 中心词索引，形状 [batch_size]
            context: 正样本索引，形状 [batch_size]
            negatives: 负样本索引，形状 [batch_size, num_negatives]
            
        返回:
            total_loss: 总损失（标量）
        """
        # ========================================================
        # 步骤1: 获取词向量
        # ========================================================
        # center_emb: [batch, embed_dim] -> [batch, 1, embed_dim]
        # 增加一个维度是为了后续的矩阵乘法
        center_emb = self.in_embed(center).unsqueeze(1)
        
        # context_emb: [batch, embed_dim] -> [batch, 1, embed_dim]
        context_emb = self.out_embed(context).unsqueeze(1)
        
        # neg_emb: [batch, num_negatives, embed_dim]
        neg_emb = self.out_embed(negatives)

        # ========================================================
        # 步骤2: 计算正样本损失（鼓励相似）
        # ========================================================
        # bmm = Batch Matrix Multiplication（批量矩阵乘法）
        # [batch, 1, dim] × [batch, dim, 1] -> [batch, 1, 1]
        # 计算中心词和上下文词的点积（相似度）
        pos_score = torch.bmm(center_emb, context_emb.transpose(1, 2)).squeeze()
        
        # 应用 log-sigmoid 函数
        # logsigmoid(x) = log(1 / (1 + e^(-x)))
        # 最大化这个值 = 最小化负值 = 损失函数
        pos_loss = -torch.nn.functional.logsigmoid(pos_score)

        # ========================================================
        # 步骤3: 计算负样本损失（鼓励不相似）
        # ========================================================
        # [batch, 1, dim] × [batch, num_neg, dim]^T -> [batch, 1, num_neg]
        # 计算中心词和每个负样本的点积
        neg_score = torch.bmm(center_emb, neg_emb.transpose(1, 2)).squeeze()
        
        # 注意负号：我们希望 neg_score 尽可能小（甚至为负）
        # logsigmoid(-x) 当 x 很大时接近 0（损失小）
        # logsigmoid(-x) 当 x 很小时很小（损失大，需要优化）
        neg_loss = -torch.nn.functional.logsigmoid(-neg_score)

        # ========================================================
        # 步骤4: 合并损失
        # ========================================================
        # pos_loss: [batch]
        # neg_loss: [batch, num_negatives]
        # sum(neg_loss, dim=1): 对每个样本的所有负样本求和 -> [batch]
        # mean(): 对整个批次求平均 -> 标量
        return torch.mean(pos_loss + torch.sum(neg_loss, dim=1))

# ============================================================
# 初始化模型和优化器
# ============================================================
embed_dim = 100  # 词向量维度
# 工业界常用配置：
# - 小规模任务（< 1万词）：50-100 维
# - 中等规模（1万-10万词）：100-200 维
# - 大规模任务（> 10万词）：200-300 维

model = SkipGramNegSampling(vocab.vocab_size, embed_dim).to(device)

# Adam 优化器：自适应学习率，是训练词向量的首选
# lr=0.003: 学习率，可能需要根据数据集大小调整
optimizer = optim.Adam(model.parameters(), lr=0.003)


## 第五步：高速训练循环

In [ ]:
# ============================================================
# 高速训练循环（工业级）
# ============================================================
epochs = 10  # 训练轮数
print(f"开始训练，总批次: {len(dataloader)}")

for epoch in range(epochs):
    start_time = time.time()  # 记录开始时间
    total_loss = 0  # 累计损失
    
    # 遍历所有批次
    for i, (center, context, negatives) in enumerate(dataloader):
        # ====================================================
        # 数据迁移到 GPU（如果可用）
        # ====================================================
        # 这一步很关键：数据必须和模型在同一设备上
        center = center.to(device)
        context = context.to(device)
        negatives = negatives.to(device)
        
        # ====================================================
        # 标准训练流程
        # ====================================================
        # 步骤1: 清空梯度
        optimizer.zero_grad()
        
        # 步骤2: 前向传播，计算损失
        loss = model(center, context, negatives)
        
        # 步骤3: 反向传播，计算梯度
        loss.backward()
        
        # 步骤4: 更新参数
        optimizer.step()
        
        # 累计损失（用于监控）
        total_loss += loss.item()
        
        # 每 100 个批次打印一次进度
        if (i+1) % 100 == 0:
            print(f"Epoch {epoch+1}, Step {i+1}, Loss: {loss.item():.4f}")

    # ========================================================
    # Epoch 结束统计
    # ========================================================
    end_time = time.time()
    epoch_time = end_time - start_time
    avg_loss = total_loss / len(dataloader)
    
    print(f"Epoch {epoch+1} 完成, 耗时: {epoch_time:.2f}s, 平均Loss: {avg_loss:.4f}")
    
    # 训练技巧：
    # - 如果 Loss 不再下降，可以降低学习率（学习率衰减）
    # - 如果 Loss 震荡，可以减小学习率或增大批次大小
    # - 通常 5-10 个 epoch 后 Loss 会趋于稳定


开始训练，总批次: 981
Epoch 1, Step 100, Loss: 3.7475
Epoch 1, Step 200, Loss: 3.0523
Epoch 1, Step 300, Loss: 2.9003
Epoch 1, Step 400, Loss: 2.8542
Epoch 1, Step 500, Loss: 2.7822
Epoch 1, Step 600, Loss: 2.6946
Epoch 1, Step 700, Loss: 2.6746
Epoch 1, Step 800, Loss: 2.6789
Epoch 1, Step 900, Loss: 2.6534
Epoch 1 完成, 耗时: 13.54s, 平均Loss: 2.9468
Epoch 2, Step 100, Loss: 2.6399
Epoch 2, Step 200, Loss: 2.6108
Epoch 2, Step 300, Loss: 2.6322
Epoch 2, Step 400, Loss: 2.6545
Epoch 2, Step 500, Loss: 2.6716
Epoch 2, Step 600, Loss: 2.6270
Epoch 2, Step 700, Loss: 2.6581
Epoch 2, Step 800, Loss: 2.6203
Epoch 2, Step 900, Loss: 2.6554
Epoch 2 完成, 耗时: 12.82s, 平均Loss: 2.6507
Epoch 3, Step 100, Loss: 2.6233
Epoch 3, Step 200, Loss: 2.6086
Epoch 3, Step 300, Loss: 2.5721
Epoch 3, Step 400, Loss: 2.5495
Epoch 3, Step 500, Loss: 2.6057
Epoch 3, Step 600, Loss: 2.5745
Epoch 3, Step 700, Loss: 2.5207
Epoch 3, Step 800, Loss: 2.5501
Epoch 3, Step 900, Loss: 2.5451
Epoch 3 完成, 耗时: 12.49s, 平均Loss: 2.5616
Epoch

## 第六步：工业级评估 (类比测试)

在工业界，我们很少看 Loss，我们看 Analogy Task (类比任务)。

比如：King - Man + Woman = `?`

In [ ]:
def get_embedding(word):
    """提取词向量
    
    注意事项：
    - 通常只使用 in_embed（中心词向量）
    - 也可以尝试 (in_embed + out_embed) / 2，有时效果更好
    - 或者只在训练时使用 out_embed，推理时丢弃
    
    参数:
        word: 要查询的词
        
    返回:
        词向量（NumPy 数组）
    """
    word_idx = vocab.word_to_idx[word]
    # 从 GPU 移回 CPU，并转换为 NumPy 数组
    return model.in_embed.weight[word_idx].cpu().detach().numpy()

def find_analogy(w1, w2, w3):
    """词类比任务（Word Analogy）
    
    这是评估词向量质量的经典方法
    
    原理：词向量的代数运算
    如果词向量学得好，应该满足：
        vec(king) - vec(man) + vec(woman) ≈ vec(queen)
        vec(Paris) - vec(France) + vec(Germany) ≈ vec(Berlin)
    
    这说明向量空间捕捉了语义关系：
    - "king - man" ≈ "皇室 - 男性" ≈ "皇室女性"
    - "+ woman" ≈ "queen"
    
    参数:
        w1, w2, w3: 三个词
        
    返回:
        最相似的词列表
        
    示例:
        find_analogy('king', 'man', 'woman')  # 期望返回 'queen'
        find_analogy('Paris', 'France', 'Germany')  # 期望返回 'Berlin'
    """
    # 步骤1: 获取三个词的向量
    v1 = get_embedding(w1)
    v2 = get_embedding(w2)
    v3 = get_embedding(w3)
    
    # 步骤2: 计算目标向量
    # 公式: vec(w1) - vec(w2) + vec(w3)
    target_vec = v1 - v2 + v3
    
    # 步骤3: 计算目标向量与所有词的相似度
    # 提取所有词向量
    all_vecs = model.in_embed.weight.cpu().detach().numpy()
    
    # 计算余弦相似度
    # 公式: cos(θ) = (A · B) / (||A|| × ||B||)
    # np.dot(all_vecs, target_vec): 所有词与目标的点积，形状 [vocab_size]
    # np.linalg.norm(all_vecs, axis=1): 所有词向量的模，形状 [vocab_size]
    # np.linalg.norm(target_vec): 目标向量的模，标量
    similarities = np.dot(all_vecs, target_vec) / (
        np.linalg.norm(all_vecs, axis=1) * np.linalg.norm(target_vec)
    )
    
    # 步骤4: 按相似度排序
    # argsort: 返回排序后的索引
    # [::-1]: 反转，从大到小
    top_indices = np.argsort(similarities)[::-1]
    
    # 步骤5: 打印结果（排除输入词）
    input_words = set([w1, w2, w3])
    found_words = []
    
    for idx in top_indices:
        word = vocab.idx_to_word[idx]
        # 跳过查询词本身
        if word not in input_words:
            found_words.append(word)
        # 找到 3 个就停止
        if len(found_words) >= 3:
            break
            
    return found_words

# ============================================================
# 测试词类比功能
# ============================================================
try:
    # 经典的词类比测试
    # 注意：需要大规模语料才能学到这种复杂的语义关系
    # 在小数据集上可能效果不明显
    print("King - Man + Woman =", find_analogy('king', 'man', 'woman'))
except Exception as e:
    print(f"词表中可能没有这些词: {e}")
    print("提示：使用真实的大规模语料库（如维基百科全文）可以获得更好的结果")

# 简单的相似词查询
try:
    # 这里的逻辑只是为了调用查找函数演示
    print("Intelligence 相关词:", find_analogy('intelligence', 'artificial', 'intelligence'))
except Exception as e:
    print(f"查询失败: {e}")

# ============================================================
# 评估建议
# ============================================================
# 工业界常用的词向量评估方法：
# 1. 词类比任务（Word Analogy）：如上所示
# 2. 词相似度任务（Word Similarity）：计算词对的相似度，与人工标注对比
# 3. 下游任务性能（Downstream Tasks）：将词向量用于分类、命名实体识别等任务
# 4. t-SNE 可视化：将高维向量降到 2D/3D 可视化，观察聚类效果


King - Man + Woman = ['vizier', 'vasa', 'filiation']
Intelligence 相似词: ['fits', 'macv', 'scientologists']
